# Panel Regression

**Master's thesis:** From Dyads to Networks: Vendor Lock-in and Systemic Vulnerability in German Public IT Procurement

**Author:** Marco Stoehr

**Notebook:** 4 of 5 (run in order; see README)

**Thesis sections:** 3.3, 4.2, Appendix B.3

## Purpose
This notebook tests the dyadic hypotheses (H1a–H3). It first completes the variable preparation required for the regression: construction of the opportunity sets (all buyer-supplier pairs active in a segment and year), the relational history variables and the leave-one-out market-structure controls. It then estimates linear probability models with buyer, supplier and year fixed effects:

1. **Segment models:** five specifications per CPV segment (Models 1–4 nested, Model 5 with the prior single-bid terms).
2. **Pooled model:** all three segments with segment-interacted fixed effects, testing whether path dependence differs between segments (H3).

## Input
- `data_analysis/`

## Main Output
- `results/`: Latex-tables with the results
- `data_analysis/supplier_lockin_profile_cpv<XX>.csv`: supplier lock-in profiles used as input for the resilience analysis
- `data_analysis/panel_cpv<XXX>.csv`: panel-data
- `data_analysis/panel_regression_<...>.csv`: results

In [1]:
# import libraries
import pandas as pd
import numpy as np
from linearmodels.panel import PanelOLS
import gc
import pyfixest as pf

In [2]:
# check version (author used 0.60.0)
pf.__version__

'0.60.0'

In [3]:
# Load raw data for segment-level activity
data = pd.read_csv('data_analysis/data_clean_final.csv')

# Extract 2-digit CPV segment
data['cpv2'] = data['CPV'].astype(str).str[:2]

# Filter to analysis period
data_post = data[data['YEAR'] >= 2017].copy()

# Active buyers and suppliers per segment-year
seg_stats = []
for cpv in ['30', '48', '72']:
    seg = data_post[data_post['cpv2'] == cpv]
    for yr in sorted(seg['YEAR'].unique()):
        yr_data = seg[seg['YEAR'] == yr]
        n_buyers = yr_data['CAE_NAME_CANONICAL'].nunique()
        n_suppliers = yr_data['WIN_NAME_FINAL'].nunique()
        n_contracts = len(yr_data)
        # Implied opportunity set size for this segment-year
        cross_product = n_buyers * n_suppliers
        seg_stats.append({
            'cpv2': cpv,
            'year': yr,
            'n_buyers': n_buyers,
            'n_suppliers': n_suppliers,
            'n_contracts': n_contracts,
            'cross_product': cross_product
        })

seg_df = pd.DataFrame(seg_stats)

# Summary per segment
for cpv in ['30', '48', '72']:
    print(f"\n=== CPV {cpv} ===")
    subset = seg_df[seg_df['cpv2'] == cpv]
    print(subset.to_string(index=False))
    total_rows = subset['cross_product'].sum()
    print(f"Total opportunity set rows: {total_rows:,.0f}")


=== CPV 30 ===
cpv2  year  n_buyers  n_suppliers  n_contracts  cross_product
  30  2017       470          393         1500         184710
  30  2018       472          378         1353         178416
  30  2019       541          470         1530         254270
  30  2020       706          605         2196         427130
  30  2021       780          741         2816         577980
  30  2022       720          612         3636         440640
  30  2023       695          595         2533         413525
Total opportunity set rows: 2,476,671

=== CPV 48 ===
cpv2  year  n_buyers  n_suppliers  n_contracts  cross_product
  48  2017       368          349          715         128432
  48  2018       375          376          662         141000
  48  2019       467          466          914         217622
  48  2020       496          519         1228         257424
  48  2021       567          580         1326         328860
  48  2022       616          670         2136         412720


### Opportunity Set Panels for each CPV-Semgment

In [4]:
def build_opportunity_panel(data_raw, dyad_years, cpv_segment):
    """
    Build expanded panel with zeros for one CPV segment.

    Logic:
    - Analysis period: 2017-2023
    - Opportunity set per year: all buyers x all suppliers active in that segment-year
    - Realized ties merged from dyad_years
    - History variables are rebuilt segment-specifically from the raw award rows,
      including the 2009-2016 observation period, and are never taken from
      dyad_years (whose history columns are pooled across CPV segments)
    - Node-level controls enter as leave-one-out measures
    """
    cpv = cpv_segment
    data_raw = data_raw.copy()
    data_raw['cpv2'] = data_raw['CPV'].astype(str).str[:2]

    seg_data = data_raw[data_raw['cpv2'] == cpv].copy()
    seg_post = seg_data[seg_data['YEAR'] >= 2017]

    # --- Step 1: Opportunity set per segment-year ---
    grids = []
    for yr in sorted(seg_post['YEAR'].unique()):
        yr_data = seg_post[seg_post['YEAR'] == yr]
        buyers = yr_data['CAE_NAME_CANONICAL'].unique()
        suppliers = yr_data['WIN_NAME_FINAL'].unique()
        grid = pd.MultiIndex.from_product(
            [buyers, suppliers, [yr]],
            names=['CAE_NAME_CANONICAL', 'WIN_NAME_FINAL', 'YEAR']
        )
        grids.append(grid.to_frame(index=False))

    panel = pd.concat(grids, ignore_index=True)
    print(f"CPV {cpv}: Opportunity set = {len(panel):,} rows")

    # --- Step 2: Mark realized ties ---
    # A dyad-year belongs to this segment if cpv_segments contains the CPV code
    dy_seg = dyad_years[
        dyad_years['cpv_segments'].astype(str).str.contains(cpv, na=False)
    ].copy()

    dy_seg['_tie'] = 1
    panel = panel.merge(
        dy_seg[['CAE_NAME_CANONICAL', 'WIN_NAME_FINAL', 'YEAR', '_tie', 'n_contracts_ijt']],
        on=['CAE_NAME_CANONICAL', 'WIN_NAME_FINAL', 'YEAR'],
        how='left'
    )
    panel['tie_ijt'] = panel['_tie'].fillna(0).astype(int)
    panel['n_contracts_ijt'] = panel['n_contracts_ijt'].fillna(0).astype(int)
    panel.drop(columns='_tie', inplace=True)

    n_expected = len(dy_seg[dy_seg['YEAR'] >= 2017])
    n_matched = int(panel['tie_ijt'].sum())
    print(f"  Realized ties: {n_matched:,} / {len(panel):,} "
          f"({panel['tie_ijt'].mean() * 100:.2f}%)")
    print(f"  Expected from dyad_years: {n_expected:,} "
          f"| unmatched: {n_expected - n_matched:,}")

    # --- Step 3: History variables from the full segment history ---
    # Distinct dyad-years observed in this segment, all years 2009-2023
    seg_all = seg_data[['CAE_NAME_CANONICAL', 'WIN_NAME_FINAL', 'YEAR']].drop_duplicates()

    hist_rows = []
    for yr in sorted(panel['YEAR'].unique()):
        prior = seg_all[seg_all['YEAR'] < yr]
        if len(prior) == 0:
            continue
        agg = prior.groupby(['CAE_NAME_CANONICAL', 'WIN_NAME_FINAL']).agg(
            cumulative_ties_ijt=('YEAR', 'nunique'),
            last_tie_year=('YEAR', 'max')
        ).reset_index()
        agg['YEAR'] = yr
        hist_rows.append(agg)

    if hist_rows:
        hist_df = pd.concat(hist_rows, ignore_index=True)
        panel = panel.merge(
            hist_df,
            on=['CAE_NAME_CANONICAL', 'WIN_NAME_FINAL', 'YEAR'],
            how='left'
        )
    else:
        panel['cumulative_ties_ijt'] = 0
        panel['last_tie_year'] = np.nan

    panel['cumulative_ties_ijt'] = panel['cumulative_ties_ijt'].fillna(0).astype(int)
    panel['ever_tied_ijt'] = (panel['cumulative_ties_ijt'] > 0).astype(int)
    panel['years_since_last_ijt'] = panel['YEAR'] - panel['last_tie_year']
    panel.drop(columns='last_tie_year', inplace=True)

    # Prior single-bid exposure: share of past award rows awarded without competition
    # Only defined for dyads with prior awards; indicator flags availability
    sb = seg_data[['CAE_NAME_CANONICAL', 'WIN_NAME_FINAL', 'YEAR', 'single_bid']].copy()
    sb = sb[sb['single_bid'].notna()]
    sb_year = sb.groupby(
        ['CAE_NAME_CANONICAL', 'WIN_NAME_FINAL', 'YEAR']
    ).agg(sb_sum=('single_bid', 'sum'), sb_n=('single_bid', 'size')).reset_index()

    sb_rows = []
    for yr in sorted(panel['YEAR'].unique()):
        prior = sb_year[sb_year['YEAR'] < yr]
        if len(prior) == 0:
            continue
        agg = prior.groupby(['CAE_NAME_CANONICAL', 'WIN_NAME_FINAL']).agg(
            sb_sum_prior=('sb_sum', 'sum'),
            sb_n_prior=('sb_n', 'sum')
        ).reset_index()
        agg['YEAR'] = yr
        sb_rows.append(agg)

    if sb_rows:
        sb_prior_df = pd.concat(sb_rows, ignore_index=True)
        panel = panel.merge(
            sb_prior_df,
            on=['CAE_NAME_CANONICAL', 'WIN_NAME_FINAL', 'YEAR'],
            how='left'
        )
    else:
        panel['sb_sum_prior'] = 0.0
        panel['sb_n_prior'] = 0.0

    panel['sb_sum_prior'] = panel['sb_sum_prior'].fillna(0.0)
    panel['sb_n_prior'] = panel['sb_n_prior'].fillna(0.0)
    panel['prior_single_bid_share_ij'] = np.where(
        panel['sb_n_prior'] > 0,
        panel['sb_sum_prior'] / panel['sb_n_prior'],
        0.0
    )
    panel['has_prior_bid_info_ij'] = (panel['sb_n_prior'] > 0).astype(int)
    panel.drop(columns=['sb_sum_prior', 'sb_n_prior'], inplace=True)

    # --- Step 4: Node-level controls ---
    # Degrees are counted from realized ties. The leave-one-out versions remove
    # the focal dyad, so that no control mechanically contains the dependent
    # variable. Only the leave-one-out versions are used in the models.
    buyer_deg = panel[panel['tie_ijt'] == 1].groupby(
        ['CAE_NAME_CANONICAL', 'YEAR']
    )['WIN_NAME_FINAL'].nunique().reset_index(name='n_suppliers_it')
    panel = panel.merge(buyer_deg, on=['CAE_NAME_CANONICAL', 'YEAR'], how='left')
    panel['n_suppliers_it'] = panel['n_suppliers_it'].fillna(0)

    supplier_deg = panel[panel['tie_ijt'] == 1].groupby(
        ['WIN_NAME_FINAL', 'YEAR']
    )['CAE_NAME_CANONICAL'].nunique().reset_index(name='n_buyers_jt')
    panel = panel.merge(supplier_deg, on=['WIN_NAME_FINAL', 'YEAR'], how='left')
    panel['n_buyers_jt'] = panel['n_buyers_jt'].fillna(0)

    total_buyers_yr = panel[panel['tie_ijt'] == 1].groupby(
        'YEAR'
    )['CAE_NAME_CANONICAL'].nunique().reset_index(name='n_active_buyers_t')
    panel = panel.merge(total_buyers_yr, on='YEAR', how='left')
    panel['supplier_share_jt'] = panel['n_buyers_jt'] / panel['n_active_buyers_t']

    panel['n_other_suppliers_it'] = (
        panel['n_suppliers_it'] - panel['tie_ijt']
    ).clip(lower=0)
    panel['n_other_buyers_jt'] = (
        panel['n_buyers_jt'] - panel['tie_ijt']
    ).clip(lower=0)
    panel['log_n_other_suppliers_it'] = np.log1p(panel['n_other_suppliers_it'])
    panel['supplier_share_excl_i_jt'] = (
        panel['n_other_buyers_jt'] / (panel['n_active_buyers_t'] - 1)
    )

    # --- Control output ---
    print(f"  Panel complete: {len(panel):,} rows, {len(panel.columns)} columns")
    print(f"  ever_tied share: {panel['ever_tied_ijt'].mean():.4f}")
    print(f"  prior single-bid info available: {panel['has_prior_bid_info_ij'].mean():.4f}")
    print("  corr with tie_ijt, raw -> leave-one-out:")
    print(f"    buyer degree:   {panel['tie_ijt'].corr(panel['n_suppliers_it']):.4f}"
          f" -> {panel['tie_ijt'].corr(panel['n_other_suppliers_it']):.4f}")
    print(f"    supplier degree:{panel['tie_ijt'].corr(panel['n_buyers_jt']):.4f}"
          f" -> {panel['tie_ijt'].corr(panel['n_other_buyers_jt']):.4f}")
    print(f"    supplier share: {panel['tie_ijt'].corr(panel['supplier_share_jt']):.4f}"
          f" -> {panel['tie_ijt'].corr(panel['supplier_share_excl_i_jt']):.4f}")

    return panel

In [5]:
dyad_years = pd.read_csv('data_analysis/dyad_years.csv')

panels = {}
for cpv in ['30', '48', '72']:
    print(f"\n{'='*60}")
    panels[cpv] = build_opportunity_panel(data, dyad_years, cpv)


CPV 30: Opportunity set = 2,476,671 rows
  Realized ties: 9,294 / 2,476,671 (0.38%)
  Expected from dyad_years: 9,294 | unmatched: 0
  Panel complete: 2,476,671 rows, 18 columns
  ever_tied share: 0.0040
  prior single-bid info available: 0.0037
  corr with tie_ijt, raw -> leave-one-out:
    buyer degree:   0.0586 -> 0.0299
    supplier degree:0.1567 -> 0.1475
    supplier share: 0.1596 -> 0.1499

CPV 48: Opportunity set = 1,917,372 rows
  Realized ties: 6,592 / 1,917,372 (0.34%)
  Expected from dyad_years: 6,592 | unmatched: 0
  Panel complete: 1,917,372 rows, 18 columns
  ever_tied share: 0.0027
  prior single-bid info available: 0.0025
  corr with tie_ijt, raw -> leave-one-out:
    buyer degree:   0.0670 -> 0.0409
    supplier degree:0.1276 -> 0.1138
    supplier share: 0.1311 -> 0.1165

CPV 72: Opportunity set = 3,914,154 rows
  Realized ties: 10,956 / 3,914,154 (0.28%)
  Expected from dyad_years: 10,956 | unmatched: 0
  Panel complete: 3,914,154 rows, 18 columns
  ever_tied share

In [6]:
# Cross-tabulate tie × ever_tied for each segment
for cpv in ['30', '48', '72']:
    p = panels[cpv]
    ct = pd.crosstab(
        p['ever_tied_ijt'], 
        p['tie_ijt'], 
        margins=True, 
        margins_name='Total'
    )
    ct.index.name = 'ever_tied'
    ct.columns.name = 'tie'
    
    # Also show conditional tie rates
    ct_pct = pd.crosstab(
        p['ever_tied_ijt'], 
        p['tie_ijt'], 
        normalize='index'
    )
    
    print(f"\n=== CPV {cpv} — Counts ===")
    print(ct)
    print(f"\n=== CPV {cpv} — P(tie=1 | ever_tied) ===")
    print(ct_pct.round(4))


=== CPV 30 — Counts ===
tie              0     1    Total
ever_tied                        
0          2459536  7329  2466865
1             7841  1965     9806
Total      2467377  9294  2476671

=== CPV 30 — P(tie=1 | ever_tied) ===
tie_ijt             0       1
ever_tied_ijt                
0              0.9970  0.0030
1              0.7996  0.2004

=== CPV 48 — Counts ===
tie              0     1    Total
ever_tied                        
0          1906564  5642  1912206
1             4216   950     5166
Total      1910780  6592  1917372

=== CPV 48 — P(tie=1 | ever_tied) ===
tie_ijt             0       1
ever_tied_ijt                
0              0.9970  0.0030
1              0.8161  0.1839

=== CPV 72 — Counts ===
tie              0      1    Total
ever_tied                         
0          3895228   8455  3903683
1             7970   2501    10471
Total      3903198  10956  3914154

=== CPV 72 — P(tie=1 | ever_tied) ===
tie_ijt             0       1
ever_tied_ijt          

In [7]:
# --- Prep: add variables still needed ---
for cpv in ['30', '48', '72']:
    p = panels[cpv]

    # Tie in the immediately preceding year.
    # Derived from years_since_last_ijt, which is built on the full segment
    # history (incl. 2009-2016); ysl == 1 is by definition a tie in t-1.
    p['tied_t_minus_1'] = (p['years_since_last_ijt'] == 1).astype(int)

    # Years since last tie, zero-filled for dyads without history.
    # Identical to ever_tied_ijt * ysl by construction, so no separate
    # interaction term is needed or identifiable.
    p['ysl_filled'] = p['years_since_last_ijt'].fillna(0)

    # Centred on the first observable value (1): zero for dyads without history
    # and for dyads tied in t-1. ever_tied_ijt then carries the level shift at
    # ysl == 1, ysl_centred the decay among previously tied dyads (H5).
    p['ysl_centred'] = (p['ysl_filled'] - p['ever_tied_ijt']).clip(lower=0)

    # Cumulative prior tie-years, centred the same way. cumulative_ties_ijt is
    # zero exactly when ever_tied_ijt is zero and at least one otherwise, so the
    # uncentred term would place ever_tied_ijt at a value that never occurs.
    p['cum_centred'] = (p['cumulative_ties_ijt'] - p['ever_tied_ijt']).clip(lower=0)

    # Dyad identifier for clustering and panel indexing
    p['dyad_id'] = p['CAE_NAME_CANONICAL'] + '___' + p['WIN_NAME_FINAL']

    panels[cpv] = p
    p.to_csv(f'data_analysis/panel_cpv{cpv}.csv', index=False)
    print(f"Saved data_analysis/panel_cpv{cpv}.csv")

# Control output
for cpv in ['30', '48', '72']:
    p = panels[cpv]
    ever = p['ever_tied_ijt'] == 1
    na_cols = p.isna().sum()
    na_cols = na_cols[na_cols > 0]
    print(f"\nCPV {cpv}: {len(p):,} rows, {len(p.columns)} columns")
    print(f"  tied_t_minus_1 share: {p['tied_t_minus_1'].mean():.5f}")
    print(f"  ties in t with tie in t-1: "
          f"{p.loc[p['tie_ijt'] == 1, 'tied_t_minus_1'].mean():.4f}")
    print(f"  ysl_centred among ever_tied: mean "
          f"{p.loc[ever, 'ysl_centred'].mean():.2f}, "
          f"max {p.loc[ever, 'ysl_centred'].max():.0f}")
    print(f"  cum_centred among ever_tied: mean "
          f"{p.loc[ever, 'cum_centred'].mean():.2f}, "
          f"max {p.loc[ever, 'cum_centred'].max():.0f}")
    print(f"  cum_centred / ysl_centred outside ever_tied: "
          f"{p.loc[~ever, ['cum_centred', 'ysl_centred']].abs().sum().sum():.0f}")
    print(f"  columns with NAs: {dict(na_cols) if len(na_cols) else 'none'}")

Saved data_analysis/panel_cpv30.csv
Saved data_analysis/panel_cpv48.csv
Saved data_analysis/panel_cpv72.csv

CPV 30: 2,476,671 rows, 23 columns
  tied_t_minus_1 share: 0.00121
  ties in t with tie in t-1: 0.0849
  ysl_centred among ever_tied: mean nan, max nan
  cum_centred among ever_tied: mean nan, max nan
  cum_centred / ysl_centred outside ever_tied: 28899
  columns with NAs: {'years_since_last_ijt': np.int64(2466865)}

CPV 48: 1,917,372 rows, 23 columns
  tied_t_minus_1 share: 0.00085
  ties in t with tie in t-1: 0.0610
  ysl_centred among ever_tied: mean 2.18, max 13
  cum_centred among ever_tied: mean 0.40, max 10
  cum_centred / ysl_centred outside ever_tied: 0
  columns with NAs: {'years_since_last_ijt': np.int64(1912206)}

CPV 72: 3,914,154 rows, 23 columns
  tied_t_minus_1 share: 0.00085
  ties in t with tie in t-1: 0.0926
  ysl_centred among ever_tied: mean 2.27, max 13
  cum_centred among ever_tied: mean 0.56, max 11
  cum_centred / ysl_centred outside ever_tied: 0
  colum

In [8]:
# LaTeX table: raw association between prior history and tie formation
rows = {}
for cpv in ['30', '48', '72']:
    d = pd.read_csv(
        f'data_analysis/panel_cpv{cpv}.csv',
        usecols=['ever_tied_ijt', 'tie_ijt'],
        dtype={'ever_tied_ijt': 'int8', 'tie_ijt': 'int8'},
    )
    g = d.groupby('ever_tied_ijt')['tie_ijt'].agg(['size', 'sum'])
    r = {'n0': int(g.loc[0, 'size']), 'k0': int(g.loc[0, 'sum']),
         'n1': int(g.loc[1, 'size']), 'k1': int(g.loc[1, 'sum'])}
    r['p0'] = r['k0'] / r['n0']
    r['p1'] = r['k1'] / r['n1']
    r['ratio'] = r['p1'] / r['p0']
    rows[cpv] = r
    del d, g
    gc.collect()

segs = ['30', '48', '72']


def _row(label, fmt, key):
    return label + ' & ' + ' & '.join(fmt(rows[s][key]) for s in segs) + r' \\'


count = lambda v: f'{v:,}'
prob = lambda v: f'{v:.4f}'
ratio = lambda v: f'{v:.1f}'

tex = '\n'.join([
    r'\begin{table}[htbp]',
    r'\centering',
    r'\begin{threeparttable}',
    r'\caption{Tie formation by prior relationship history, 2017--2023}',
    r'\label{tab:crosstab}',
    r'\begin{tabular}{lccc}',
    r'\toprule',
    ' & ' + ' & '.join(f'CPV {s}' for s in segs) + r' \\',
    r'\midrule',
    r'\multicolumn{4}{l}{\textit{Dyads without prior tie}} \\',
    _row(r'\quad Dyad-years at risk', count, 'n0'),
    _row(r'\quad Ties formed', count, 'k0'),
    _row(r'\quad $P(\text{tie})$', prob, 'p0'),
    r'\addlinespace',
    r'\multicolumn{4}{l}{\textit{Dyads with prior tie}} \\',
    _row(r'\quad Dyad-years at risk', count, 'n1'),
    _row(r'\quad Ties formed', count, 'k1'),
    _row(r'\quad $P(\text{tie})$', prob, 'p1'),
    r'\midrule',
    _row(r'Ratio of probabilities', ratio, 'ratio'),
    r'\bottomrule',
    r'\end{tabular}',
    r'\begin{tablenotes}\small\item Opportunity sets for the analysis period. '
    r'A dyad counts as having a prior tie if the same buyer and supplier were '
    r'linked by an award in the same CPV segment in any earlier year since '
    r'2009. Unadjusted proportions; no controls or fixed effects.'
    r'\end{tablenotes}',
    r'\end{threeparttable}',
    r'\end{table}',
])

print(tex)
with open('results/table_crosstab.tex', 'w', encoding='utf-8') as f:
    f.write(tex)
print('\nSaved results/table_crosstab.tex')

\begin{table}[htbp]
\centering
\begin{threeparttable}
\caption{Tie formation by prior relationship history, 2017--2023}
\label{tab:crosstab}
\begin{tabular}{lccc}
\toprule
 & CPV 30 & CPV 48 & CPV 72 \\
\midrule
\multicolumn{4}{l}{\textit{Dyads without prior tie}} \\
\quad Dyad-years at risk & 2,466,865 & 1,912,206 & 3,903,683 \\
\quad Ties formed & 7,329 & 5,642 & 8,455 \\
\quad $P(\text{tie})$ & 0.0030 & 0.0030 & 0.0022 \\
\addlinespace
\multicolumn{4}{l}{\textit{Dyads with prior tie}} \\
\quad Dyad-years at risk & 9,806 & 5,166 & 10,471 \\
\quad Ties formed & 1,965 & 950 & 2,501 \\
\quad $P(\text{tie})$ & 0.2004 & 0.1839 & 0.2389 \\
\midrule
Ratio of probabilities & 67.4 & 62.3 & 110.3 \\
\bottomrule
\end{tabular}
\begin{tablenotes}\small\item Opportunity sets for the analysis period. A dyad counts as having a prior tie if the same buyer and supplier were linked by an award in the same CPV segment in any earlier year since 2009. Unadjusted proportions; no controls or fixed effects.\

# Panel-Regression Models


In [9]:
MODEL_COLS = [
    'CAE_NAME_CANONICAL', 'WIN_NAME_FINAL', 'YEAR', 'tie_ijt',
    'ever_tied_ijt', 'cum_centred', 'ysl_centred',
    'supplier_share_excl_i_jt', 'log_n_other_suppliers_it',
    'prior_single_bid_share_ij', 'has_prior_bid_info_ij',
]

MODEL_DTYPES = {
    'CAE_NAME_CANONICAL': 'category',
    'WIN_NAME_FINAL': 'category',
    'YEAR': 'int16',
    'tie_ijt': 'float32',
    'ever_tied_ijt': 'float32',
    'cum_centred': 'float32',
    'ysl_centred': 'float32',
    'supplier_share_excl_i_jt': 'float32',
    'log_n_other_suppliers_it': 'float32',
    'prior_single_bid_share_ij': 'float32',
    'has_prior_bid_info_ij': 'float32',
}

SPECS = {
    # H1: does any prior tie raise the probability of a tie in t?
    'M1': "tie_ijt ~ ever_tied_ijt",

    # + does the number of prior tie-years add anything beyond their existence?
    'M2': "tie_ijt ~ ever_tied_ijt + cum_centred",

    # H5: + does the effect decay with each year since the last award?
    'M3': "tie_ijt ~ ever_tied_ijt + cum_centred + ysl_centred",

    # + market structure controls, leave-one-out
    'M4': ("tie_ijt ~ ever_tied_ijt + cum_centred + ysl_centred"
           " + supplier_share_excl_i_jt + log_n_other_suppliers_it"),

    # H2 at dyad level: is path dependence stronger where past awards ran
    # without competition?
    'M5': ("tie_ijt ~ ever_tied_ijt + ever_x_sbshare + has_prior_bid_info_ij"
           " + cum_centred + ysl_centred"
           " + supplier_share_excl_i_jt + log_n_other_suppliers_it"),
}

# Buyer, supplier and year fixed effects; SEs clustered on the buyer
FE = " | CAE_NAME_CANONICAL + WIN_NAME_FINAL + YEAR"


def tidy_result(res, model, label):
    """Extract coefficients so the fitted object can be released."""
    out = pd.DataFrame({
        'estimate': res.coef(),
        'se': res.se(),
        'tstat': res.tstat(),
        'pval': res.pvalue(),
    })
    out.index.name = 'term'
    out = out.reset_index()
    out.insert(0, 'model', model)
    out.insert(0, 'segment', label)
    out['n_obs'] = res._N
    out['r2_within'] = res._r2_within
    return out


coef_tables = []

for cpv in ['30', '48', '72']:
    print(f"\n{'=' * 60}")
    p = pd.read_csv(
        f'data_analysis/panel_cpv{cpv}.csv',
        usecols=MODEL_COLS,
        dtype=MODEL_DTYPES,
    )

    p['ever_x_sbshare'] = (
        p['ever_tied_ijt'] * p['prior_single_bid_share_ij']
    ).astype('float32')

    print(f"CPV {cpv}: {len(p):,} rows, "
          f"{p.memory_usage(deep=True).sum() / 1e9:.2f} GB in memory")

    for name, rhs in SPECS.items():
        res = pf.feols(rhs + FE, data=p, vcov={"CRV1": "CAE_NAME_CANONICAL"})
        print(f"\nCPV {cpv} — {name}")
        print(res.summary())
        coef_tables.append(tidy_result(res, name, cpv))
        del res
        gc.collect()

    del p
    gc.collect()

coefs = pd.concat(coef_tables, ignore_index=True)
coefs.to_csv('data_analysis/panel_regression_results.csv', index=False)
print(f"\nSaved data_analysis/panel_regression_results.csv "
      f"({len(coefs)} coefficient rows)")


CPV 30: 2,476,671 rows, 0.10 GB in memory

CPV 30 — M1
###

Estimation:  OLS
Dep. var.: tie_ijt, Fixed effects: CAE_NAME_CANONICAL + WIN_NAME_FINAL + YEAR
sample: None = all
Inference:  CRV1
Observations:  2476671

| Coefficient   |   Estimate |   Std. Error |   t value |   Pr(>|t|) |   2.5% |   97.5% |
|:--------------|-----------:|-------------:|----------:|-----------:|-------:|--------:|
| ever_tied_ijt |      0.177 |        0.006 |    29.333 |      0.000 |  0.165 |   0.189 |
---
RMSE: 0.059 R2: 0.055 R2 Within: 0.032 
None

CPV 30 — M2
###

Estimation:  OLS
Dep. var.: tie_ijt, Fixed effects: CAE_NAME_CANONICAL + WIN_NAME_FINAL + YEAR
sample: None = all
Inference:  CRV1
Observations:  2476671

| Coefficient   |   Estimate |   Std. Error |   t value |   Pr(>|t|) |   2.5% |   97.5% |
|:--------------|-----------:|-------------:|----------:|-----------:|-------:|--------:|
| ever_tied_ijt |      0.146 |        0.005 |    26.855 |      0.000 |  0.135 |   0.156 |
| cum_centred   |     

In [10]:
# H2: is path dependence stronger in some segments? Panels stacked, fixed
# effects interacted with segment, CPV 30 as reference.
# Can be run on its own after a kernel restart; needs only the imports
# and MODEL_DTYPES.
POOL_COLS = [
    'CAE_NAME_CANONICAL', 'WIN_NAME_FINAL', 'YEAR', 'tie_ijt',
    'ever_tied_ijt', 'cum_centred', 'ysl_centred',
    'supplier_share_excl_i_jt', 'log_n_other_suppliers_it',
]

pooled = []
for cpv in ['30', '48', '72']:
    q = pd.read_csv(
        f'data_analysis/panel_cpv{cpv}.csv',
        usecols=POOL_COLS,
        dtype={c: MODEL_DTYPES[c] for c in POOL_COLS},
    )
    q['cpv'] = cpv
    pooled.append(q)

pooled = pd.concat(pooled, ignore_index=True)
pooled['cpv'] = pooled['cpv'].astype('category')
for col in ['CAE_NAME_CANONICAL', 'WIN_NAME_FINAL']:
    pooled[col] = pooled[col].astype('category')
gc.collect()

pooled['ever_x_cpv48'] = (
    pooled['ever_tied_ijt'] * (pooled['cpv'] == '48')
).astype('float32')
pooled['ever_x_cpv72'] = (
    pooled['ever_tied_ijt'] * (pooled['cpv'] == '72')
).astype('float32')

print(f"Pooled panel: {len(pooled):,} rows, "
      f"{pooled.memory_usage(deep=True).sum() / 1e9:.2f} GB in memory")
print(pooled.groupby('cpv', observed=True)['tie_ijt'].agg(['size', 'mean']))

m_pooled = pf.feols(
    "tie_ijt ~ ever_tied_ijt + ever_x_cpv48 + ever_x_cpv72"
    " + cum_centred + ysl_centred"
    " + supplier_share_excl_i_jt + log_n_other_suppliers_it"
    " | CAE_NAME_CANONICAL^cpv + WIN_NAME_FINAL^cpv + YEAR^cpv",
    data=pooled,
    vcov={"CRV1": "CAE_NAME_CANONICAL"},
)

print("\nPooled — H2 (segment interaction)")
print(m_pooled.summary())

pooled_coefs = tidy_result(m_pooled, 'H2_pooled', 'all')
pooled_coefs.to_csv('data_analysis/panel_regression_h2.csv', index=False)
print("\nSaved data_analysis/panel_regression_h2.csv")

del pooled
gc.collect()

Pooled panel: 8,308,197 rows, 0.32 GB in memory
        size      mean
cpv                   
30   2476671  0.003753
48   1917372  0.003438
72   3914154  0.002799

Pooled — H2 (segment interaction)
###

Estimation:  OLS
Dep. var.: tie_ijt, Fixed effects: CAE_NAME_CANONICAL^cpv + WIN_NAME_FINAL^cpv + YEAR^cpv
sample: None = all
Inference:  CRV1
Observations:  8308197

| Coefficient              |   Estimate |   Std. Error |   t value |   Pr(>|t|) |   2.5% |   97.5% |
|:-------------------------|-----------:|-------------:|----------:|-----------:|-------:|--------:|
| ever_tied_ijt            |      0.178 |        0.006 |    27.972 |      0.000 |  0.165 |   0.190 |
| ever_x_cpv48             |     -0.013 |        0.009 |    -1.413 |      0.158 | -0.030 |   0.005 |
| ever_x_cpv72             |      0.040 |        0.016 |     2.476 |      0.013 |  0.008 |   0.072 |
| cum_centred              |      0.071 |        0.005 |    14.432 |      0.000 |  0.061 |   0.080 |
| ysl_centred           

466

In [11]:
# Diagnostic: is the null result for the market-structure controls in CPV 72
# substantive, or is there simply no within-supplier variation left after the
# supplier fixed effects?
for cpv in ['30', '48', '72']:
    d = pd.read_csv(
        f'data_analysis/panel_cpv{cpv}.csv',
        usecols=['WIN_NAME_FINAL', 'YEAR', 'tie_ijt', 'supplier_share_excl_i_jt'],
        dtype={'WIN_NAME_FINAL': 'category', 'YEAR': 'int16',
               'tie_ijt': 'float32', 'supplier_share_excl_i_jt': 'float32'},
    )
    tied = d[d['tie_ijt'] == 1]

    # One row per supplier-year
    sy = tied.groupby(['WIN_NAME_FINAL', 'YEAR'], observed=True)[
        'supplier_share_excl_i_jt'].first().reset_index()
    years_per_supplier = sy.groupby('WIN_NAME_FINAL', observed=True).size()
    within_sd = sy.groupby('WIN_NAME_FINAL', observed=True)[
        'supplier_share_excl_i_jt'].std()

    print(f"CPV {cpv}: {len(years_per_supplier):,} suppliers with ties")
    print(f"  observed in >=2 years: {(years_per_supplier >= 2).mean():.3f}")
    print(f"  mean years per supplier: {years_per_supplier.mean():.2f}")
    print(f"  mean within-supplier SD of share: {within_sd.mean():.5f}")

    del d, tied, sy
    gc.collect()

CPV 30: 2,250 suppliers with ties
  observed in >=2 years: 0.300
  mean years per supplier: 1.69
  mean within-supplier SD of share: 0.00168
CPV 48: 2,355 suppliers with ties
  observed in >=2 years: 0.273
  mean years per supplier: 1.52
  mean within-supplier SD of share: 0.00157
CPV 72: 4,257 suppliers with ties
  observed in >=2 years: 0.277
  mean years per supplier: 1.54
  mean within-supplier SD of share: 0.00101


In [12]:
# Build LaTeX tables from the saved coefficient files
coefs = pd.read_csv('data_analysis/panel_regression_results.csv',
                    dtype={'segment': str, 'model': str, 'term': str})
pooled_coefs = pd.read_csv('data_analysis/panel_regression_h2.csv',
                           dtype={'segment': str, 'model': str, 'term': str})
coefs = pd.concat([coefs, pooled_coefs], ignore_index=True)

TERM_LABELS = [
    ('ever_tied_ijt',            r'Prior tie (ever)'),
    ('ever_x_cpv48',             r'Prior tie $\times$ CPV 48'),
    ('ever_x_cpv72',             r'Prior tie $\times$ CPV 72'),
    ('ever_x_sbshare',           r'Prior tie $\times$ prior single-bid share'),
    ('has_prior_bid_info_ij',    r'Prior bid information available'),
    ('cum_centred',              r'Prior tie-years (centred)'),
    ('ysl_centred',              r'Years since last tie (centred)'),
    ('supplier_share_excl_i_jt', r'Supplier reach (excl.\ $i$)'),
    ('log_n_other_suppliers_it', r'Buyer diversification (log, excl.\ $j$)'),
]


def _stars(p):
    return '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else ''


def latex_table(columns, caption, label, note):
    """columns: list of (segment, model, header)."""
    lookup = {(r.segment, r.model, r.term): r for r in coefs.itertuples()}
    n_col = len(columns)

    out = [
        r'\begin{table}[htbp]',
        r'\centering',
        r'\begin{threeparttable}',
        rf'\caption{{{caption}}}',
        rf'\label{{{label}}}',
        r'\begin{tabular}{l' + 'c' * n_col + '}',
        r'\toprule',
        ' & ' + ' & '.join(h for _, _, h in columns) + r' \\',
        r'\midrule',
    ]

    for term, label_txt in TERM_LABELS:
        if not any((s, m, term) in lookup for s, m, _ in columns):
            continue
        est, se = [], []
        for seg, mod, _ in columns:
            r = lookup.get((seg, mod, term))
            est.append(f'{r.estimate:.3f}{_stars(r.pval)}' if r is not None else '')
            se.append(f'({r.se:.3f})' if r is not None else '')
        out.append(f'{label_txt} & ' + ' & '.join(est) + r' \\')
        out.append(' & ' + ' & '.join(se) + r' \\')

    n_obs, r2 = [], []
    for seg, mod, _ in columns:
        rows = coefs[(coefs['segment'] == seg) & (coefs['model'] == mod)]
        n_obs.append(f'{int(rows["n_obs"].iloc[0]):,}' if len(rows) else '')
        if len(rows) and 'r2_within' in rows.columns:
            r2.append(f'{rows["r2_within"].iloc[0]:.3f}')
        else:
            r2.append('')

    out += [
        r'\midrule',
        r'Observations & ' + ' & '.join(n_obs) + r' \\',
    ]
    if any(r2):
        out.append(r'Within $R^2$ & ' + ' & '.join(r2) + r' \\')
    out += [
        r'\bottomrule',
        r'\end{tabular}',
        rf'\begin{{tablenotes}}\small\item {note}\end{{tablenotes}}',
        r'\end{threeparttable}',
        r'\end{table}',
    ]
    return '\n'.join(out)


NOTE_MAIN = (
    r'Linear probability models. Buyer, supplier and year fixed effects; '
    r'in the pooled model all fixed effects are interacted with the CPV '
    r'segment. Standard errors in parentheses, clustered on the contracting '
    r'authority. Baseline tie rates: 0.375\% (CPV 30), 0.344\% (CPV 48), '
    r'0.280\% (CPV 72). $^{*}p<0.05$, $^{**}p<0.01$, $^{***}p<0.001$.'
)

NOTE_NESTED = (
    r'Linear probability models for CPV segment {seg}. Buyer, supplier and '
    r'year fixed effects; standard errors in parentheses, clustered on the '
    r'contracting authority. $^{{*}}p<0.05$, $^{{**}}p<0.01$, '
    r'$^{{***}}p<0.001$.'
)

main = latex_table(
    columns=[
        ('30', 'M4', 'CPV 30'),
        ('48', 'M4', 'CPV 48'),
        ('72', 'M4', 'CPV 72'),
        ('all', 'H2_pooled', 'Pooled'),
    ],
    caption='Determinants of tie formation, full specification',
    label='tab:panel-main',
    note=NOTE_MAIN,
)

print(main)

with open('results/table_panel_main.tex', 'w', encoding='utf-8') as f:
    f.write(main)

for seg in ['30', '48', '72']:
    tex = latex_table(
        columns=[(seg, m, m) for m in ['M1', 'M2', 'M3', 'M4', 'M5']],
        caption=f'Nested specifications, CPV {seg}',
        label=f'tab:panel-nested-{seg}',
        note=NOTE_NESTED.format(seg=seg),
    )
    with open(f'results/table_panel_nested_{seg}.tex', 'w',
              encoding='utf-8') as f:
        f.write(tex)

print('\nSaved table_panel_main.tex and three nested tables to results/')

\begin{table}[htbp]
\centering
\begin{threeparttable}
\caption{Determinants of tie formation, full specification}
\label{tab:panel-main}
\begin{tabular}{lcccc}
\toprule
 & CPV 30 & CPV 48 & CPV 72 & Pooled \\
\midrule
Prior tie (ever) & 0.183*** & 0.171*** & 0.209*** & 0.178*** \\
 & (0.007) & (0.011) & (0.021) & (0.006) \\
Prior tie $\times$ CPV 48 &  &  &  & -0.013 \\
 &  &  &  & (0.009) \\
Prior tie $\times$ CPV 72 &  &  &  & 0.040* \\
 &  &  &  & (0.016) \\
Prior tie-years (centred) & 0.060*** & 0.060*** & 0.082*** & 0.071*** \\
 & (0.006) & (0.007) & (0.007) & (0.005) \\
Years since last tie (centred) & -0.015*** & -0.015*** & -0.013*** & -0.014*** \\
 & (0.001) & (0.002) & (0.004) & (0.002) \\
Supplier reach (excl.\ $i$) & 0.621*** & 0.574*** & -0.013 & 0.474*** \\
 & (0.047) & (0.056) & (0.088) & (0.043) \\
Buyer diversification (log, excl.\ $j$) & -0.011*** & -0.020*** & -0.001 & -0.007*** \\
 & (0.001) & (0.002) & (0.001) & (0.001) \\
\midrule
Observations & 2,476,671 & 1,917,

## Extraction of the suppliers with the strongest lock-in effects

In [13]:
# Construct supplier-level lock-in profiles per segment
PROFILE_COLS = [
    'CAE_NAME_CANONICAL', 'WIN_NAME_FINAL', 'tie_ijt',
    'ever_tied_ijt', 'cumulative_ties_ijt', 'n_suppliers_it',
]

PROFILE_DTYPES = {
    'CAE_NAME_CANONICAL': 'category',
    'WIN_NAME_FINAL': 'category',
    'tie_ijt': 'int8',
    'ever_tied_ijt': 'int8',
    'cumulative_ties_ijt': 'int16',
    'n_suppliers_it': 'float32',
}

for cpv in ['30', '48', '72']:
    p = pd.read_csv(
        f'data_analysis/panel_cpv{cpv}.csv',
        usecols=PROFILE_COLS,
        dtype=PROFILE_DTYPES,
    )

    # Aggregate over the whole analysis period rather than a single snapshot,
    # so the profile reflects the full relationship structure
    realized = p[p['tie_ijt'] == 1]

    # A buyer counts as locked in to j if the dyad already had a shared history
    locked_in = realized[realized['ever_tied_ijt'] == 1]

    # Breadth: number of locked-in buyers
    breadth = locked_in.groupby('WIN_NAME_FINAL', observed=True)[
        'CAE_NAME_CANONICAL'].nunique().reset_index(name='n_locked_in_buyers')

    # Depth: mean prior tie-years across locked-in buyers
    depth = locked_in.groupby('WIN_NAME_FINAL', observed=True)[
        'cumulative_ties_ijt'].mean().reset_index(name='mean_cumulative_ties')

    # Captive buyers: locked-in buyers sourcing from a single supplier that year
    captive = locked_in[locked_in['n_suppliers_it'] == 1].groupby(
        'WIN_NAME_FINAL', observed=True)[
        'CAE_NAME_CANONICAL'].nunique().reset_index(name='n_captive_buyers')

    # All buyers per supplier, for normalisation
    total_buyers = realized.groupby('WIN_NAME_FINAL', observed=True)[
        'CAE_NAME_CANONICAL'].nunique().reset_index(name='n_total_buyers')

    profile = (total_buyers
               .merge(breadth, on='WIN_NAME_FINAL', how='left')
               .merge(depth, on='WIN_NAME_FINAL', how='left')
               .merge(captive, on='WIN_NAME_FINAL', how='left'))

    profile['n_locked_in_buyers'] = profile['n_locked_in_buyers'].fillna(0).astype(int)
    profile['n_captive_buyers'] = profile['n_captive_buyers'].fillna(0).astype(int)
    profile['captive_share'] = profile['n_captive_buyers'] / profile['n_total_buyers']
    profile['lock_in_pct'] = profile['n_locked_in_buyers'] / profile['n_total_buyers']

    profile = profile.sort_values('n_locked_in_buyers', ascending=False)
    profile.to_csv(f'data_analysis/supplier_lockin_profile_cpv{cpv}.csv', index=False)

    print(f"\n=== CPV {cpv} — Top 15 by locked-in buyers ===")
    print(profile.head(15).to_string(index=False))
    print(f"\nTotal suppliers: {len(profile):,}")
    print(f"Suppliers with at least one captive buyer: "
          f"{(profile['n_captive_buyers'] > 0).sum():,}")

    del p, realized, locked_in
    gc.collect()


=== CPV 30 — Top 15 by locked-in buyers ===
                                    WIN_NAME_FINAL  n_total_buyers  n_locked_in_buyers  mean_cumulative_ties  n_captive_buyers  captive_share  lock_in_pct
                                        Bechtle AG             479                 175              2.474227                63       0.131524     0.365344
                                         Cancom SE             296                  69              1.720000                26       0.087838     0.233108
                            Ricoh Deutschland GmbH             166                  62              2.000000                22       0.132530     0.373494
                           Lyreco Deutschland GmbH             127                  43              1.903226                23       0.181102     0.338583
                        Computacenter AG & Co. oHG              87                  33              2.626866                10       0.114943     0.379310
                         

In [14]:
# How distinct is the lock-in criterion from plain supplier size?
for cpv in ['30', '48', '72']:
    prof = pd.read_csv(f'data_analysis/supplier_lockin_profile_cpv{cpv}.csv')
    r_p = prof['n_locked_in_buyers'].corr(prof['n_total_buyers'])
    r_s = prof['n_locked_in_buyers'].corr(prof['n_total_buyers'], method='spearman')
    top_lock = set(prof.nlargest(20, 'n_locked_in_buyers')['WIN_NAME_FINAL'])
    top_size = set(prof.nlargest(20, 'n_total_buyers')['WIN_NAME_FINAL'])
    print(f"CPV {cpv}: Pearson {r_p:.3f}, Spearman {r_s:.3f}, "
          f"top-20 overlap {len(top_lock & top_size)}/20")

CPV 30: Pearson 0.936, Spearman 0.519, top-20 overlap 15/20
CPV 48: Pearson 0.869, Spearman 0.329, top-20 overlap 14/20
CPV 72: Pearson 0.887, Spearman 0.355, top-20 overlap 16/20


In [15]:
# Check actor overlap across segments
overlap_raw = pd.read_csv(
    'data_analysis/data_clean_final.csv',
    usecols=['CAE_NAME_CANONICAL', 'WIN_NAME_FINAL', 'YEAR', 'CPV'],
    dtype={'CAE_NAME_CANONICAL': 'category', 'WIN_NAME_FINAL': 'category'},
)
overlap_raw = overlap_raw[overlap_raw['YEAR'] >= 2017].copy()
overlap_raw['cpv2'] = overlap_raw['CPV'].astype(str).str[:2]
overlap_raw = overlap_raw[overlap_raw['cpv2'].isin(['30', '48', '72'])]

for role, col in [('Buyers', 'CAE_NAME_CANONICAL'),
                  ('Suppliers', 'WIN_NAME_FINAL')]:
    n_segments = overlap_raw.groupby(col, observed=True)['cpv2'].nunique()
    total = len(n_segments)
    print(f"\n{role}: {total:,} unique actors")
    for k in [1, 2, 3]:
        n = (n_segments == k).sum()
        print(f"  active in exactly {k} segment(s): {n:,} ({n / total * 100:.1f}%)")

del overlap_raw
gc.collect()


Buyers: 4,412 unique actors
  active in exactly 1 segment(s): 2,968 (67.3%)
  active in exactly 2 segment(s): 818 (18.5%)
  active in exactly 3 segment(s): 626 (14.2%)

Suppliers: 7,620 unique actors
  active in exactly 1 segment(s): 6,554 (86.0%)
  active in exactly 2 segment(s): 890 (11.7%)
  active in exactly 3 segment(s): 176 (2.3%)


0